In [1]:
!pip install docx2txt


In [3]:
!pip install pywin32

In [5]:
!pip install pdfplumber

     ---------------------------------------- 0.0/43.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/43.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/43.6 kB ? eta -:--:--
     --------- ------------------------------ 10.2/43.6 kB ? eta -:--:--
     --------- ------------------------------ 10.2/43.6 kB ? eta -:--:--
     ------------------ -------------------- 20.5/43.6 kB 81.9 kB/s eta 0:00:01
     ------------------ -------------------- 20.5/43.6 kB 81.9 kB/s eta 0:00:01
     ----------------------------------- -- 41.0/43.6 kB 140.3 kB/s eta 0:00:01
     -------------------------------------- 43.6/43.6 kB 133.4 kB/s eta 0:00:00
     ---------------------------------------- 0.0/68.1 kB ? eta -:--:--
     ---------------------------------------- 0.0/68.1 kB ? eta -:--:--
     ------------------------ --------------- 41.0/68.1 kB 2.0 MB/s eta 0:00:01
     ------------------------ --------------- 41.0/68.1 kB 2.0 MB/s eta 0:00:01
     ---------

In [1]:
import os
import re
import pandas as pd
import numpy as np

from PyPDF2 import PdfReader
from docx import Document

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report


In [2]:
data_path = r"C:\Users\leela\Downloads\P643_DATASET\Resumes"


In [3]:
def extract_text_from_pdf(file_path):
    text = ""
    try:
        reader = PdfReader(file_path)
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
    except:
        pass
    return text


def extract_text_from_docx(file_path):
    text = ""
    try:
        doc = Document(file_path)
        for para in doc.paragraphs:
            text += para.text + " "
    except:
        pass
    return text


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


In [5]:
texts = []
labels = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        
        if file.lower().endswith((".pdf", ".docx", ".doc")):
            
            file_path = os.path.join(root, file)

            relative_path = os.path.relpath(root, data_path)

            

            if relative_path == ".":
                label = "React Developer"
            else:
                raw_label = relative_path.split(os.sep)[0].lower()
                cleaned = raw_label.replace(" resumes", "").replace(" lightning insight", "").strip()

                if cleaned == "sql developer":
                    label = "SQL Developer"
                elif "developer" not in cleaned:
                    label = cleaned.title() + " Developer"
                else:
                    label = cleaned.title()


            # Extract text
            if file.lower().endswith(".pdf"):
                text = extract_text_from_pdf(file_path)
            else:
                text = extract_text_from_docx(file_path)

            text = clean_text(text)

            texts.append(text)
            labels.append(label)

print("Total resumes loaded:", len(texts))


Total resumes loaded: 79


In [6]:
clean_texts = []
clean_labels = []

for t, l in zip(texts, labels):
    if len(t.strip()) > 100:   # keep meaningful resumes only
        clean_texts.append(t)
        clean_labels.append(l)

print("Final usable resumes:", len(clean_texts))


Final usable resumes: 53


In [7]:
df = pd.DataFrame({
    "resume_text": clean_texts,
    "category": clean_labels
})

df.head()


,resume_text,category
0,name ravali p curriculum vitae specialization ...,React Developer
1,susovan bag seeking a challenging position in ...,React Developer
2,kanumuru deepak reddy career objective to secu...,React Developer
3,haripriya battina experience as ui developer i...,React Developer
4,kamalakar reddy a linked in https www linkedin...,React Developer


In [8]:
df.shape

(53, 2)

In [9]:
print(df["category"].value_counts())



React Developer         21
SQL Developer           11
Workday Developer       11
Peoplesoft Developer    10
Name: category, dtype: int64


In [10]:
df.head()

,resume_text,category
0,name ravali p curriculum vitae specialization ...,React Developer
1,susovan bag seeking a challenging position in ...,React Developer
2,kanumuru deepak reddy career objective to secu...,React Developer
3,haripriya battina experience as ui developer i...,React Developer
4,kamalakar reddy a linked in https www linkedin...,React Developer


In [11]:
from collections import Counter

all_words = " ".join(df["resume_text"]).split()
word_counts = Counter(all_words)

print(word_counts.most_common(20))


[('and', 1399), ('the', 766), ('in', 707), ('to', 630), ('of', 508), ('on', 375), ('for', 360), ('experience', 313), ('using', 247), ('with', 227), ('as', 221), ('workday', 194), ('application', 187), ('a', 180), ('sql', 178), ('project', 177), ('from', 175), ('server', 169), ('data', 167), ('reports', 164)]


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

X = df["resume_text"]
y = df["category"]


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42,stratify=y)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)



In [14]:
from sklearn.linear_model import LogisticRegression

logist_model = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='lbfgs',
    max_iter=500
)
logist_model.fit(X_train, y_train)


LogisticRegression(max_iter=500)

In [15]:

y_pred = logist_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy: 0.9090909090909091

Classification Report:

                      precision    recall  f1-score   support

Peoplesoft Developer       1.00      1.00      1.00         2
     React Developer       0.83      1.00      0.91         5
       SQL Developer       1.00      0.50      0.67         2
   Workday Developer       1.00      1.00      1.00         2

            accuracy                           0.91        11
           macro avg       0.96      0.88      0.89        11
        weighted avg       0.92      0.91      0.90        11



In [16]:
from sklearn.naive_bayes import MultinomialNB
naive_model = MultinomialNB(alpha=0.5)
naive_model.fit(X_train,y_train)


MultinomialNB(alpha=0.5)

In [17]:
y_naive_predict=naive_model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_naive_predict))
print("Classification Report:")
print(classification_report(y_test,y_naive_predict))
      

Accuracy: 1.0
Classification Report:
                      precision    recall  f1-score   support

Peoplesoft Developer       1.00      1.00      1.00         2
     React Developer       1.00      1.00      1.00         5
       SQL Developer       1.00      1.00      1.00         2
   Workday Developer       1.00      1.00      1.00         2

            accuracy                           1.00        11
           macro avg       1.00      1.00      1.00        11
        weighted avg       1.00      1.00      1.00        11



In [18]:
from sklearn.svm import LinearSVC
svm_model = LinearSVC(
    C=1.0,
    penalty='l2',
    loss='squared_hinge',
    max_iter=2000
)
svm_model.fit(X_train,y_train)


D:\Users\leela\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


LinearSVC(max_iter=2000)

In [19]:
y_svm_predict=svm_model.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, y_svm_predict))
print("\nClassification Report:\n")
print(classification_report(y_test, y_svm_predict))

SVM Accuracy: 1.0

Classification Report:

                      precision    recall  f1-score   support

Peoplesoft Developer       1.00      1.00      1.00         2
     React Developer       1.00      1.00      1.00         5
       SQL Developer       1.00      1.00      1.00         2
   Workday Developer       1.00      1.00      1.00         2

            accuracy                           1.00        11
           macro avg       1.00      1.00      1.00        11
        weighted avg       1.00      1.00      1.00        11



In [20]:
from sklearn.model_selection import cross_val_score

svm_scores = cross_val_score(LinearSVC(C=1.0,
    penalty='l2',
    loss='squared_hinge',
    max_iter=2000), X_train, y_train, cv=5)
print("SVM Cross Validation Accuracy:", svm_scores.mean())


SVM Cross Validation Accuracy: 0.9527777777777778


D:\Users\leela\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
D:\Users\leela\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
D:\Users\leela\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
D:\Users\leela\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
D:\Users\leela\anaconda3\Lib\site-packages\sklearn\svm\_clas

In [21]:


naive_scores = cross_val_score(MultinomialNB(alpha=0.5), X_train, y_train, cv=5)
print("naive Cross Validation Accuracy:", naive_scores.mean())


naive Cross Validation Accuracy: 0.9527777777777778


Both Linear SVM and Multinomial Naive Bayes achieved 100% test accuracy and approximately 95% cross-validation accuracy. However, Linear SVM is generally more robust for high-dimensional TF-IDF features, so we  considered it as the best performing model in this case.

In [23]:
import joblib

joblib.dump(svm_model, "resume_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']